In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt
from scipy.optimize import minimize
from scipy.optimize import linprog
from scipy.optimize import LinearConstraint
import math
from sklearn.linear_model import LinearRegression
import yfinance as yf
from datetime import datetime, timedelta
import random

def get_option_data(df, date_val, exdate_val):
    
    filtered_df = df[(df['date'] == int(date_val)) & (df['exdate'] == int(exdate_val))]
    result_df = filtered_df[['strike_price', 'cp_flag', 'best_bid', 'best_offer']]
    return result_df.to_numpy()

# _spx_cache_df = pd.read_csv('spx_prices.csv', index_col=0, parse_dates=True)
# _spx_cache = {
#         int(pd.to_datetime(d).strftime('%Y%m%d')): float(v) 
#         for d, v in _spx_cache_df['Close'].to_dict().items()
#     }

# def get_spx_close(date_val):
#     try:
#         d_int = int(date_val)
#     except (ValueError, TypeError):
#         return None

#     # Direct lookup
#     if d_int in _spx_cache:
#         return _spx_cache[d_int]

#     available_dates = sorted(_spx_cache.keys())
#     past_dates = [d for d in available_dates if d <= d_int]

#     if past_dates:
#         return _spx_cache[past_dates[-1]]

#     return None

def get_spx_close(date_val):
    
    date_str = str(date_val)
    start_date = datetime.strptime(date_str, '%Y%m%d')
 
    end_date = start_date + timedelta(days=1)

    spx = yf.download('^SPX', start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), progress=False)

    if not spx.empty:
        return spx['Close'].iloc[0]
    else:
        return None

def process_date_exdate(df, date_val, exdate_val):
    
    test = get_option_data(df, date_val, exdate_val)

    test_calls = test[test[:, 1] == 'C']
    test_puts = test[test[:, 1] == 'P']
    
    test_calls = np.delete(test_calls, 1, axis=1).astype(float)
    test_puts = np.delete(test_puts, 1, axis=1).astype(float)
    
    common_strikes = np.intersect1d(test_calls[:, 0], test_puts[:, 0])
    
    test_calls = test_calls[np.isin(test_calls[:, 0], common_strikes)]
    
    test_puts = test_puts[np.isin(test_puts[:, 0], common_strikes)]
    
    test_calls = test_calls[test_calls[:, 0].argsort()]
    test_puts = test_puts[test_puts[:, 0].argsort()]

    mid_calls = test_calls[:, 1:3].mean(axis=1)
    mid_puts = test_puts[:, 1:3].mean(axis=1)
    
    diff_prices = (mid_calls - mid_puts).reshape(-1, 1)

    test_avg = np.hstack((test_calls[:, [0]], diff_prices))
    
    X = test_avg[:, 0].reshape(-1, 1)
    y = test_avg[:, 1]
    
    model = LinearRegression()
    model.fit(X, y)
    
    B = model.coef_[0]
    A = model.intercept_

    y_pred = model.predict(X)

    lower_bound = test_calls[:, 1] - test_puts[:, 2]
    upper_bound = test_calls[:, 2] - test_puts[:, 1]

    within_bounds = (y_pred >= lower_bound) & (y_pred <= upper_bound)
    
    num_violations = np.sum(~within_bounds)
    print(f"Checking regression constraints for {len(X)} strikes:")
    print(f"Number of violations: {num_violations}")
    
    if num_violations > 0:
        print("\nViolations found at these strikes:")
        violations_df = pd.DataFrame({
            'Strike': X.flatten()[~within_bounds],
            'Lower Bound': lower_bound[~within_bounds],
            'Predicted': y_pred[~within_bounds],
            'Upper Bound': upper_bound[~within_bounds]
        })
        display(violations_df)
    else:
        print("All predicted values are within the specified bounds.")

    D = -B
    F = A/D

    spot = get_spx_close(date_val)
    spot_val = float(spot.iloc[0]) if hasattr(spot, 'iloc') else float(spot)

    strike_scale = spot_val / F
    price_scale = spot_val / (D * F)
    
    test_calls[:, 0] = test_calls[:, 0] * strike_scale
    test_calls[:, 1:3] = test_calls[:, 1:3] * price_scale
    
    test_puts[:, 0] = test_puts[:, 0] * strike_scale
    test_puts[:, 1:3] = test_puts[:, 1:3] * price_scale

    K = test_calls[:, 0]

    C_h = np.minimum(test_calls[:, 2], test_puts[:, 2] + spot_val - K)

    C_l = np.maximum(test_calls[:, 1], test_puts[:, 1] + spot_val - K)

    return K, C_l, C_h, spot_val

def load_theta(S, K, C_h, C_l, end_buf):
    N = len(K)
    A = np.zeros((N+N+1,N))
    b = np.zeros((N+N+1,1))
    lb = np.zeros((N,1))
    ub = np.zeros((N,1))
    L = 0
    Kbar = K[-1] + end_buf
    
    
    
    # for i in range(1,N-1):
    #     A[N-1+i-1,i-1] = -1.0/(K[i]-K[i-1])
    #     A[N-1+i-1,i] = 1.0/(K[i]-K[i-1]) + 1.0/(K[i+1]-K[i])
    #     A[N-1+i-1,i+1] = -1.0/(K[i+1]-K[i])
    
    # A[N-1+N-2+1,0] = 1
    # b[N-1+N-2+1,0] = S - L
    
    # A[N-1+N-2+2,N-1] = -1
    
    # A[N-1+N-2+3,0] = 1.0/(K[0] - L) + 1.0/(K[1]-K[0])
    # A[N-1+N-2+3,1] = -1.0/(K[1]-K[0])
    # b[N-1+N-2+3,0] = S/(K[0] - L)
    
    # A[N-1+N-2+4,N-2] = -1.0/(K[N-1]-K[N-2])
    # A[N-1+N-2+4,N-1] = 1.0/(Kbar - K[N-1]) + 1.0/(K[N-1]-K[N-2])

    for i in range(1,N-1):
        A[i-1,i-1] = -1.0/(K[i]-K[i-1])
        A[i-1,i] = 1.0/(K[i]-K[i-1]) + 1.0/(K[i+1]-K[i])
        A[i-1,i+1] = -1.0/(K[i+1]-K[i])

    for i in range(0,N-1):
        A[N-1+i,i] = -1.0
        A[N-1+i,i+1] = 1.0

    for i in range(0,N):
        lb[i] = max([0.0,S-K[i],C_l[i]])
        ub[i] = min([S,C_h[i]])
    
    A[N-2+N-2,N-1] = -1

    A[N-2+N-2+1,0] = -1
    b[N-2+N-2+1,0] = K[0] - S

    A[N-2+N-2+2,0] = 1.0/(K[0] - L) + 1.0/(K[1]-K[0])
    A[N-2+N-2+2,1] = -1.0/(K[1]-K[0])
    b[N-2+N-2+2,0] = (S - L)/(K[0] - L)
    
    A[N-2+N-2+3,N-2] = -1.0/(K[N-1]-K[N-2])
    A[N-2+N-2+3,N-1] = 1.0/(Kbar - K[N-1]) + 1.0/(K[N-1]-K[N-2])

    A[N-2+N-2+4,0] = 1
    b[N-2+N-2+4,0] = S - L
    
    A = np.concatenate((A, np.identity(N), -np.identity(N)), axis=0)
    b = np.concatenate((b, ub, -lb), axis=0)    
    
    b = b.T[0]

    n = A.shape[1] 

    C = (lb + ub/2).flatten()
    # res = linprog(list(c), list(A), b)

    # if not res.success:
    #     print("Initial point failure:", res.message)
    #     return None, None, None, None

    # def objective(x):
    #     # Calculate the slack for each inequality
    #     slack = b - A @ x
    #     return -math.prod(slack)

    # x0 = res.x
    
    # result = minimize(objective, x0, method='trust-constr', constraints=LinearConstraint(A, lb=-np.inf, ub=b))
    
    # if result.success:
    #     analytic_center = result.x
    # else:
    #     print("Optimization failed:", result.message)
    #     return None, None, None, None
    
    E = 1

    def augmented_log(val, eps):
        condition_true = np.log(np.maximum(val, eps))
        condition_false = np.log(eps) + (1/eps) * (val - eps)
        return np.where(val >= eps, condition_true, condition_false)
    
    def objective(x):
        slack = b - A @ x
        return -np.sum(augmented_log(slack, E))

    def grad_objective(x):
        slack = b - A @ x
        d_aug_log = np.where(slack >= E, 1.0 / slack, 1.0 / E)
        return A.T @ d_aug_log

    for i in range(10):
        result = minimize(objective, C, method='BFGS', jac=grad_objective)
        C = result.x
        E /= 10

    # print(np.min(b - A @ C))

    # index = ((b - A @ C) < 0).argmax()

    # plt.scatter(K[index - 1: index + 2], lb[index - 1: index + 2].flatten(), c='red', s=5)
    # plt.scatter(K[index - 1: index + 2], C[index - 1: index + 2], c='blue', s=5)
    # plt.scatter(K[index - 1: index + 2], ub[index - 1: index + 2].flatten(), c='red', s=5)
    # plt.plot([K[index-1], K[index+1]], [C[index - 1], C[index + 1]])
    # plt.show()

    def Atilde(sigma,A,B,w,K):
        return 0.5*(A-sigma*B)*np.exp(-(w-K)/sigma) + 0.5*(A+sigma*B)*np.exp((w-K)/sigma)

    def Btilde(sigma,A,B,w,K):
        return -(0.5/sigma)*(A-sigma*B)*np.exp(-(w-K)/sigma) + (0.5/sigma)*(A+sigma*B)*np.exp((w-K)/sigma)

    def findSigmaHat(A,B,w,K1,K2,V):
        my_eps = 1.0e-8
        def SigmaHat_obj(sigma):
            return Atilde(sigma,A,B,w,K1) + Btilde(sigma,A,B,w,K1)*(K2-w) - V
        low = my_eps
        high = 1.0
        while (SigmaHat_obj(high) > 0):
            high = 2.0*high
        while (high - low > my_eps):
            mid = 0.5 * (low + high)
            if mid == low or mid == high: return mid
            if (SigmaHat_obj(mid) >= 0):
                low = mid
            else:
                high = mid
        return 0.5 * (low + high)

    def SigmaTilde(sigma,A,B,w,K1,K2,V):
        #sigma_h = findSigmaHat(A,B,w,K1,K2,V)
        my_eps = 1.0e-8
        At = Atilde(sigma,A,B,w,K1)
        Bt = Btilde(sigma,A,B,w,K1)
        def SigmaTilde_obj(sigma_t):
            return 0.5*(At+sigma_t*Bt)*np.exp((K2-w)/sigma_t) + 0.5*(At-sigma_t*Bt)*np.exp(-(K2-w)/sigma_t) - V
        low = my_eps
        high = 1
        while (SigmaTilde_obj(high) > 0):
            high = 2.0*high
        while (high - low > my_eps):
            mid = 0.5 * (low + high)
            if mid == low or mid == high: return mid
            if (SigmaTilde_obj(mid) >= 0):
                low = mid
            else:
                high = mid
        return 0.5 * (low + high)

    def FindSigma(A,B,w,K1,K2,V,B1,sigma_h):
        #sigma_h = findSigmaHat(A,B,w,K1,K2,V)
        my_eps = 1.0e-8
        def FindSigma_obj(sigma):
            sigma_t = SigmaTilde(sigma,A,B,w,K1,K2,V)
            At = Atilde(sigma,A,B,w,K1)
            Bt = Btilde(sigma,A,B,w,K1)
            return B1 - (0.5/sigma_t)*(At + sigma_t*Bt)*np.exp((K2-w)/sigma_t) + (0.5/sigma_t)*(At - sigma_t*Bt)*np.exp(-(K2-w)/sigma_t)
        low = sigma_h
        high = sigma_h+1
        while (FindSigma_obj(high) > 0):
            high = 2.0*high
        #print(high)
        while (high - low > my_eps):
            mid = 0.5 * (low + high)
            if mid == low or mid == high: return mid
            if (FindSigma_obj(mid) >= 0):
                low = mid
            else:
                high = mid
        return 0.5 * (low + high)

    i0 = 0
    for i in range(N-1):
        if (K[i]<S) & (S<K[i+1]):
            i0 = i
    
    Lt1 = C[i0] + (S-K[i0])*(C[i0]-C[i0-1])/(K[i0]-K[i0-1])
    Lt2 = C[i0+1] + (S-K[i0+1])*(C[i0+2]-C[i0+1])/(K[i0+2]-K[i0+1])
    Lt = max([Lt1,Lt2])
    
    Ut = C[i0]*(K[i0+1]-S)/(K[i0+1]-K[i0]) + C[i0+1]*(S-K[i0])/(K[i0+1]-K[i0])
    
    delta = 0.5
    Vtemp = delta*Lt + (1-delta)*Ut
    
    K1 = [0]
    V1 = [0]
    for i in range(N):
        if (K[i]<S):
            K1.append(K[i])
            V1.append(C[i]-S+K[i])
    
    K2 = [0]
    V2 = [0]
    for i in range(N):
        if (K[N-1-i]>S):
            K2.append(Kbar - K[N-1-i])
            V2.append(C[N-1-i])
    
    K1.append(S)
    K2.append(Kbar-S)
    V1.append(Vtemp)
    V2.append(Vtemp)

    def Interpolate(K,V):
        N = len(K)
        nu = []
        kappa = []
        omega = []
        sigmas = []
        A = V[0]
        delta = 0.5
        #B = delta*(V[1]-V[0])/(K[1]-K[0]) + (1-delta)*V[0]/K[0]
        B = delta*(V[1]-V[0])/(K[1]-K[0])
        for i in range(N-2):
            #print(i)
            K1t = K[i]
            K2t = K[i+1]
            V1t = V[i]
            V2t = V[i+1]
            B1 = delta*(V[i+2]-V2t)/(K[i+2]-K2t) + (1.0-delta)*(V2t-V1t)/(K2t-K1t)
            w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
            sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
            #print(sigma_h)
            sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
            #print(sigma)
            sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
            At = Atilde(sigma,A,B,w,K1t)
            Bt = Btilde(sigma,A,B,w,K1t)
            nu.append(K1t)
            nu.append(w)
            omega.append(1.0/sigma)
            omega.append(1.0/sigma_t)
            kappa.append([0.5*(A+sigma*B),0.5*(A-sigma*B)])
            kappa.append([0.5*(At+sigma_t*Bt),0.5*(At-sigma_t*Bt)])
            sigmas.append(sigma);
            sigmas.append(sigma_t);
            A = V2t
            #B = (0.5/sigma_t)*(At+sigma_t*Bt)*np.exp((K2-w)/sigma_t) - (0.5/sigma_t)*(At-sigma_t*Bt)*np.exp(-(K2-w)/sigma_t)
            B=B1
        return nu, sigmas, B
        
    nu1, sigmas1, B1_last = Interpolate(K1,V1)
    nu2, sigmas2, B2_last = Interpolate(K2,V2)

    delta1 = 0.5
    rho = delta1 + delta1*(V2[-2]-V2[-1])/(K2[-1]-K2[-2]) + (1.0-delta1)*(V1[-1]-V1[-2])/(K1[-1]-K1[-2])

    K1t = K1[-2]
    K2t = K1[-1]
    V1t = V1[-2]
    V2t = V1[-1]
    A = V1t
    B = B1_last
    B1 = rho
    w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
    sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
    sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
    sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
    At = Atilde(sigma,A,B,w,K1t)
    Bt = Btilde(sigma,A,B,w,K1t)
    
    nu1.append(K1t)
    nu1.append(w)
    sigmas1.append(sigma)
    sigmas1.append(sigma_t)

    K1t = K2[-2]
    K2t = K2[-1]
    V1t = V2[-2]
    V2t = V2[-1]
    A = V1t
    B = B2_last
    B1 = 1 - rho
    w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
    sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
    sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
    sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
    At = Atilde(sigma,A,B,w,K1t)
    Bt = Btilde(sigma,A,B,w,K1t)
    
    nu2.append(K1t)
    nu2.append(w)
    sigmas2.append(sigma)
    sigmas2.append(sigma_t)

    initial_sigs1 = np.array(sigmas1)
    initial_sigs2 = np.array(sigmas2[::-1])
    initial_nus1 = np.array(nu1)
    initial_nus2 = Kbar - np.array(nu2[::-1])

    return initial_nus1, initial_sigs1, initial_nus2, initial_sigs2

In [2]:
def load_c_values(R1, R2, S0, theta):
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2
    nus1_full = np.concatenate([nus1, [S0]])
    nus2_full = np.concatenate([[S0], nus2])
    c_v1_unit = []
    dist0 = nus1_full[1] - nus1_full[0]
    c_v1_unit.append((-np.exp(-dist0/sigs1[0]), np.exp(dist0/sigs1[0])))
    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1_full[j+2] - nus1_full[j+1]
        c_v1_unit.append((0.5 * (P - sigs1[j+1]*S) * np.exp(-dist/sigs1[j+1]), 0.5 * (P + sigs1[j+1]*S) * np.exp(dist/sigs1[j+1])))
    c_v2_unit = [None] * R2
    dist_u = nus2_full[-1] - nus2_full[-2]
    c_v2_unit[-1] = (np.exp(dist_u/sigs2[-1]), -np.exp(-dist_u/sigs2[-1]))
    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2_full[j] - nus2_full[j-1]
        c_v2_unit[j-1] = (0.5 * (P - sigs2[j-1]*S) * np.exp(dist/sigs2[j-1]), 0.5 * (P + sigs2[j-1]*S) * np.exp(-dist/sigs2[j-1]))
    v1, Dk_v1 = c_v1_unit[-1][0] + c_v1_unit[-1][1], (1/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])
    v2, Dk_v2 = c_v2_unit[0][0] + c_v2_unit[0][1], (1/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])
    denom = (Dk_v1 * v2 - v1 * Dk_v2)
    lam1, lam2 = v2 / denom, v1 / denom
    
    c_v1 = np.array(c_v1_unit) * lam1
    c_v2 = np.array(c_v2_unit) * lam2
    return c_v1, c_v2

def validate_and_check_market(R1, R2, S0, theta, bid_ask_arr):
    eps = 1e-7
    
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    try:
        c_v1, c_v2 = load_c_values(R1, R2, S0, theta)
    except:
        return False

    K_vec = bid_ask_arr[:, 0]
    nus1_full = np.concatenate([nus1, [S0]])
    nus2_full = np.concatenate([[S0], nus2])
    mask_left = K_vec <= S0
    mask_right = ~mask_left
    results = np.zeros_like(K_vec)

    if np.any(mask_left):
        K_left = K_vec[mask_left]
        indices = np.searchsorted(nus1_full, K_left, side='left') - 1
        indices = np.clip(indices, 0, R1 - 1)
        dist = nus1_full[indices + 1] - K_left
        results[mask_left] = (c_v1[indices, 0] * np.exp(dist/sigs1[indices]) + c_v1[indices, 1] * np.exp(-dist/sigs1[indices])) + S0 - K_left

    if np.any(mask_right):
        K_right = K_vec[mask_right]
        indices = np.searchsorted(nus2_full, K_right, side='left') - 1
        indices = np.clip(indices, 0, R2 - 1)
        dist = K_right - nus2_full[indices]
        results[mask_right] = (c_v2[indices, 0] * np.exp(-dist/sigs2[indices]) + c_v2[indices, 1] * np.exp(dist/sigs2[indices]))

    return np.all((results >= bid_ask_arr[:, 1]) & (results <= bid_ask_arr[:, 2]))

def validate_and_check_market_withoutbound(R1, R2, S0, theta, bid_ask_arr):
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    try:
        c_v1, c_v2 = load_c_values(R1, R2, S0, theta)
    except:
        return False

    K_vec = bid_ask_arr[:, 0]
    nus1_full = np.concatenate([nus1, [S0]])
    nus2_full = np.concatenate([[S0], nus2])
    mask_left = K_vec <= S0
    mask_right = ~mask_left
    results = np.zeros_like(K_vec)

    if np.any(mask_left):
        K_left = K_vec[mask_left]
        indices = np.searchsorted(nus1_full, K_left, side='left') - 1
        indices = np.clip(indices, 0, R1 - 1)
        dist = nus1_full[indices + 1] - K_left
        results[mask_left] = (c_v1[indices, 0] * np.exp(dist/sigs1[indices]) + c_v1[indices, 1] * np.exp(-dist/sigs1[indices])) + S0 - K_left

    if np.any(mask_right):
        K_right = K_vec[mask_right]
        indices = np.searchsorted(nus2_full, K_right, side='left') - 1
        indices = np.clip(indices, 0, R2 - 1)
        dist = K_right - nus2_full[indices]
        results[mask_right] = (c_v2[indices, 0] * np.exp(-dist/sigs2[indices]) + c_v2[indices, 1] * np.exp(dist/sigs2[indices]))

    return np.all((results >= bid_ask_arr[:, 1]) & (results <= bid_ask_arr[:, 2]))

def Objective(sigs):
    return np.log(np.sum(np.diff(sigs)**2))

def coordinateDescent(R1, R2, S0, init_theta, bid_ask_arr, limit = None, num_iterations = 1, num_indices = 1, target_left = None, target_right = None):
    theta = init_theta.copy()

    if not validate_and_check_market_withoutbound(R1, R2, S0, theta, bid_ask_arr): 
        print("Doesn't satisfy!!!!")

    sigs1 = theta[R1 : 2 * R1]
    sigs2 = theta[2 * R1 + R2 :]
    sigs = np.concatenate([sigs1, sigs2])

    current_loss = Objective(sigs)
    loss_check = Objective(sigs)

    rand_perm = np.random.permutation(R1 + R2 - num_indices + 1)

    for i in range(num_iterations * (R1 + R2 - num_indices + 1)):

        grad_sigs = np.zeros_like(sigs)

        if (i % 100 == 99):
            if loss_check - Objective(sigs) < limit: break
            else: loss_check = Objective(sigs)

        s_prev = sigs[:-2]
        s_next = sigs[2:]

        target = (s_prev + s_next) / 2
        grad_sigs[1:-1] = target - sigs[1:-1]

        if target_left == None: grad_sigs[0] = (sigs[1] - sigs[0])
        else: grad_sigs[0] = (target_left - sigs[0])
        
        if target_right == None: grad_sigs[-1] = (sigs[-2] - sigs[-1])
        else: grad_sigs[-1] = (target_right - sigs[-1])

        start = i % (R1 + R2 - num_indices + 1)
        indices = range(start, start+num_indices)

        update_vector = np.zeros_like(theta)
        for index in indices:
            if grad_sigs[index] == 0:
                continue
            if index < R1:
                update_vector[R1 + index] += grad_sigs[index]
            else:
                update_vector[2 * R1 + R2 + (index - R1)] += grad_sigs[index]

        curr_alpha = 1.0
        for b in range(5):
            test_theta = theta + curr_alpha * update_vector
            test_sigs = np.concatenate([test_theta[R1 : 2 * R1], test_theta[2 * R1 + R2 :]])
            new_loss = Objective(test_sigs)
            if validate_and_check_market(R1, R2, S0, test_theta, bid_ask_arr) and new_loss < current_loss:
                theta = test_theta
                current_loss = new_loss
                sigs1 = theta[R1 : 2 * R1]
                sigs2 = theta[2 * R1 + R2 :]
                sigs = np.concatenate([sigs1, sigs2])
                break
            curr_alpha *= 0.5
    return theta
    
def coordinateDescentSpecific(R1, R2, S0, init_theta, bid_ask_arr, I, num_indices = 1, target_left = None, target_right = None):
    theta = init_theta.copy()

    if not validate_and_check_market_withoutbound(R1, R2, S0, theta, bid_ask_arr): 
        print("Doesn't satisfy!!!!")
    
    sigs1 = theta[R1 : 2 * R1]
    sigs2 = theta[2 * R1 + R2 :]
    sigs = np.concatenate([sigs1, sigs2])

    current_loss = Objective(sigs)

    grad_sigs = np.zeros_like(sigs)

    s_prev = sigs[:-2]
    s_next = sigs[2:]

    target = (s_prev + s_next) / 2
    grad_sigs[1:-1] = target - sigs[1:-1]

    if target_left == None: grad_sigs[0] = (sigs[1] - sigs[0])
    else: grad_sigs[0] = (target_left - sigs[0])
    
    if target_right == None: grad_sigs[-1] = (sigs[-2] - sigs[-1])
    else: grad_sigs[-1] = (target_right - sigs[-1])

    start = max(I - (num_indices)//2, 0)
    end = min(I + (num_indices+1)//2, len(grad_sigs))
    indices = range(start, end)

    update_vector = np.zeros_like(theta)
    for index in indices:
        if grad_sigs[index] == 0:
            continue
        if index < R1:
            update_vector[R1 + index] += grad_sigs[index]
        else:
            update_vector[2 * R1 + R2 + (index - R1)] += grad_sigs[index]

    curr_alpha = 1.0
    for b in range(2):
        test_theta = theta + curr_alpha * update_vector
        test_sigs = np.concatenate([test_theta[R1 : 2 * R1], test_theta[2 * R1 + R2 :]])
        if validate_and_check_market(R1, R2, S0, test_theta, bid_ask_arr) and Objective(test_sigs) < current_loss:
            theta = test_theta
            break
        curr_alpha *= 0.5
    return theta

def calculate_J(R1, R2, S0, theta, c_v1, c_v2):
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2
    nus1_full = np.concatenate([nus1, [S0]])
    nus2_full = np.concatenate([[S0], nus2])
    jumps = []
    for j in range(R1 - 1):
        v_left = (1/sigs1[j]**2) * (c_v1[j, 0] + c_v1[j, 1])
        dist = nus1_full[j+2] - nus1_full[j+1]
        v_right = (1/sigs1[j+1]**2) * (c_v1[j+1, 0]*np.exp(dist/sigs1[j+1]) + c_v1[j+1, 1]*np.exp(-dist/sigs1[j+1]))
        jumps.append(v_right - v_left)
    v1_S0 = (1/sigs1[-1]**2) * (c_v1[-1, 0] + c_v1[-1, 1])
    v2_S0 = (1/sigs2[0]**2) * (c_v2[0, 0] + c_v2[0, 1])
    jumps.append(v2_S0 - v1_S0)
    for j in range(R2 - 1):
        v_right = (1/sigs2[j+1]**2) * (c_v2[j+1, 0] + c_v2[j+1, 1])
        dist = nus2_full[j+1] - nus2_full[j]
        v_left = (1/sigs2[j]**2) * (c_v2[j, 0]*np.exp(-dist/sigs2[j]) + c_v2[j, 1]*np.exp(dist/sigs2[j]))
        jumps.append(v_right - v_left)
    return np.sum(np.array(jumps) ** 2)

def optimize_nus(R1, R2, S0, init_theta, bid_ask_arr, num_additions = 100, tol=1e-5, target_left=None, target_right=None):
    theta = init_theta.copy()
    sigs = np.concatenate([theta[R1:2*R1], theta[2*R1+R2:]])
    nus = np.concatenate([theta[:R1], [S0], theta[2*R1:2*R1+R2]])

    min_length = np.min(np.diff(nus))

    for i in range(num_additions):

        index = np.argmax(np.diff(nus))

        if ((nus[index+1] - nus[index])/2) < min_length: break

        new_nu = (nus[index+1] + nus[index])/2
        nus = np.insert(nus, index + 1, new_nu)

        if (index < R1):
            nus = np.delete(nus, R1 + 1)
            R1 += 1
        else:
            nus = np.delete(nus, R1)
            R2 += 1

        sigs = np.insert(sigs, index + 1, sigs[index])

        theta = np.concatenate([nus[:R1], sigs[:R1], nus[R1:], sigs[R1:]])
        theta = coordinateDescentSpecific(R1, R2, S0, theta, bid_ask_arr, index+1, num_indices = (R1+R2)//10, target_left=target_left, target_right=target_right)

        sigs = np.concatenate([theta[R1:2*R1], theta[2*R1+R2:]])
        nus = np.concatenate([theta[:R1], [S0], theta[2*R1:2*R1+R2]])

    return R1, R2, theta

In [3]:
def plot_V_over_C2(R1, R2, theta, S0, ax=None, label=None, color=None):
    nus = np.concatenate([theta[:R1], [S0], theta[2*R1:2*R1+R2]])
    sigs = np.concatenate([theta[R1:2*R1], theta[2*R1+R2:]])

    y_vals = (sigs**2)

    if ax is None:
        fig, ax = plt.subplots(figsize=(15, 4))

    j = 0

    for i in range(len(nus) - 1):
        if (nus[i] > 250):
            ax.hlines(y_vals[i], np.log(nus[i]/S0), np.log(nus[i+1]/S0), lw=0.5, label=label if j == 0 else "", color=color)
            j += 1
            if i < len(nus) - 2:
                ax.vlines(np.log(nus[i+1]/S0), min(y_vals[i], y_vals[i+1]), max(y_vals[i], y_vals[i+1]), lw = 0.5, color=color)

def run_optimizer(initial_nus1, initial_sigs1, initial_nus2, initial_sigs2, S0, market_bid_ask, ax=None, label=None, color=None, target_left=None, target_right=None):
    R1 = len(initial_sigs1)
    R2 = len(initial_sigs2)

    init_theta = np.concatenate([initial_nus1, initial_sigs1, initial_nus2, initial_sigs2])
    init_sigs = np.concatenate([initial_sigs1, initial_sigs2])

    # plot_V_over_C2(R1, R2, init_theta, S0, ax=ax, label=label, color=color)

    opt_theta = coordinateDescent(R1, R2, S0, init_theta, market_bid_ask, limit = 0.01, num_iterations=1000, num_indices=1, target_left=target_left, target_right=target_right)
    opt_sigs = np.concatenate([opt_theta[R1 : 2 * R1], opt_theta[2 * R1 + R2 :]])
    
    new_R1, new_R2, refined_theta = optimize_nus(R1, R2, S0, opt_theta, market_bid_ask, num_additions=1000, target_left=target_left, target_right=target_right)
    refined_sigs = np.concatenate([refined_theta[new_R1 : 2 * new_R1], refined_theta[2 * new_R1 + new_R2 :]])

    final_theta = refined_theta
    final_sigs = refined_sigs

    init = Objective(init_sigs)
    final = Objective(final_sigs)
    print(init, final, 100*(1-np.exp(final-init)))

    plot_V_over_C2(new_R1, new_R2, final_theta, S0, ax=ax, label=label, color=color)

    return opt_sigs[0], opt_sigs[-1]

def test_index(INDEX, dataframe, end_buf):
    unique_date_exdate_combinations = dataframe[['date', 'exdate']].drop_duplicates()
    date = unique_date_exdate_combinations.iloc[INDEX, 0]
    exdate = unique_date_exdate_combinations.iloc[INDEX, 1]
    print("Date: ", date, " ExDate: ", exdate)
    K, C_l, C_h, S0 = process_date_exdate(dataframe, date, exdate)
    market_bid_ask = np.column_stack([K, C_l, C_h])
    initial_nus1, initial_sigs1, initial_nus2, initial_sigs2 = load_theta(S0, K, C_h, C_l, end_buf)
    print(initial_sigs1, initial_sigs2)
    if initial_nus1 is None: return
    run_optimizer(initial_nus1, initial_sigs1, initial_nus2, initial_sigs2, S0, market_bid_ask)
    plt.show()

In [4]:
import warnings
import itertools
warnings.filterwarnings('ignore')

dataframe = pd.read_csv("SPX_opt_2011_2012.csv")

test_index(2, dataframe, 400)
# end_buf sets Kbar to last nu + end_buf